# Run inference for EarTTS

In [ ]:
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

# No need to manually register the model anymore!
# It's automatically registered via the config registry

engine_args = AsyncEngineArgs(
    model="eartts_vllm_model",
    dtype="bfloat16",
    max_model_len=256,
    gpu_memory_utilization=0.8,
    enable_prompt_embeds=True,
    return_hidden_states=True,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using embeddings directly
    #load_format="dummy",
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=20, temperature=1.0)

# embeddings for context phase
inputs = {"prompt_embeds": torch.randn(1, 1152)}
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id="1"):
    print(output, flush=True)
    hidden_states = output.outputs[0].hidden_states
    print(f"hidden_states num: {len(hidden_states)}, shape: {hidden_states[0].shape}", flush=True)
    break